# HuBERT Fine-tuning v2 — Turkish MBT-C Emotion (PZ / NG)

Phase 2.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")


In [ ]:
CONFIG = {
    "AUDIO_ZIP":   "/content/drive/MyDrive/CHANGE_ME/segments_filtered.zip",
    "DATASET_CSV": "/content/drive/MyDrive/CHANGE_ME/labeling_outputs/dataset_final.csv",
    "OUTPUT_DIR":  "/content/drive/MyDrive/CHANGE_ME/finetune_outputs",

    "TRAIN_SOURCE": "ours",      # "ours" = dataset_final.csv train split | "turev" = TurEV-DB (acted Turkish)
    "BASE_MODEL": "utter-project/mHuBERT-147",   # "facebook/hubert-base-ls960", "facebook/hubert-large-ll60k"
    "RUN_CV": False,             # True = grouped 5-fold session-disjoint CV on our full labeled set (~4-6 h)

    "SEED": 42,
    "VAL_FRACTION": 0.15,
    "EPOCHS": 12,
    "LR": 3e-5,
    "BATCH_TRAIN": 16,
    "BATCH_EVAL": 8,
    "TRAIN_CROP_S": 12.0,
    "EVAL_MAX_S": 25.0,
    "AUGMENT": True,
    "EARLY_STOP_PATIENCE": 3,
}
TAG = f"{CONFIG['TRAIN_SOURCE']}-{CONFIG['BASE_MODEL'].split('/')[-1]}"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
if CONFIG["OUTPUT_DIR"].startswith("/content/drive") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive not mounted but OUTPUT_DIR is on Drive — run Cell 1.")
print("Config OK — run tag:", TAG, "| CV:", CONFIG["RUN_CV"])

In [ ]:
%pip -q install "transformers>=4.56,<5" "accelerate>=1.0" "huggingface_hub<1.0" librosa soundfile scikit-learn pandas
print("Installs done.")

In [ ]:
import re, glob, json, random, warnings, zipfile, subprocess
import numpy as np, pandas as pd, librosa
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
warnings.filterwarnings("ignore")

np.random.seed(CONFIG["SEED"]); random.seed(CONFIG["SEED"]); torch.manual_seed(CONFIG["SEED"])

def _fix_zip_name(info):
    if info.flag_bits & 0x800: return info.filename
    try: return info.filename.encode("cp437").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError): return info.filename

AUDIO_ROOT = "/content/segments_extracted"
marker = os.path.join(AUDIO_ROOT, ".extract_done")
if not os.path.exists(marker):
    print("Extracting audio zip to local disk...")
    os.makedirs(AUDIO_ROOT, exist_ok=True)
    with zipfile.ZipFile(CONFIG["AUDIO_ZIP"]) as zf:
        for info in tqdm(zf.infolist(), desc="unzip"):
            if info.is_dir(): continue
            dest = os.path.join(AUDIO_ROOT, _fix_zip_name(info).lstrip("/"))
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with zf.open(info) as src, open(dest, "wb") as out:
                out.write(src.read())
    open(marker, "w").write("ok")

path_by_id = {Path(p).stem: p for p in glob.glob(os.path.join(AUDIO_ROOT, "**", "*.wav"), recursive=True)}
print(f"{len(path_by_id)} wavs available")

LABEL2ID = {"NG": 0, "PZ": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df = pd.read_csv(CONFIG["DATASET_CSV"], dtype={"id": str})
df["path"] = df["id"].map(path_by_id)
missing = int(df["path"].isna().sum())
if missing: print(f"WARNING: {missing} labeled clips not found in the zip — dropped.")
df = df.dropna(subset=["path"]).copy()
df["label"] = df["final_label"].map(LABEL2ID)
if "weight" not in df.columns: df["weight"] = 1.0
df["session"] = df.get("session", pd.Series(index=df.index, dtype=object)).fillna(df["id"])

test_df = df[df["split"] == "test"].reset_index(drop=True)

if CONFIG["TRAIN_SOURCE"] == "ours":
    pool = df[df["split"] == "train"].reset_index(drop=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG["VAL_FRACTION"], random_state=CONFIG["SEED"])
    tr_ix, va_ix = next(gss.split(pool, groups=pool["session"]))
    train_df, val_df = pool.iloc[tr_ix].reset_index(drop=True), pool.iloc[va_ix].reset_index(drop=True)
    assert set(train_df["session"]) & set(val_df["session"]) == set(), "session leakage!"
elif CONFIG["TRAIN_SOURCE"] == "turev":
    TUREV_DIR = "/content/TurEV-DB"
    if not os.path.isdir(TUREV_DIR):
        print("Cloning TurEV-DB (Canpolat et al. 2020, github.com/Xeonen/TurEV-DB)...")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Xeonen/TurEV-DB", TUREV_DIR], check=True)
    TMAP = {"angry": 0, "kizgin": 0, "sad": 0, "uzgun": 0, "happy": 1, "mutlu": 1}
    rows = []
    for p in glob.glob(TUREV_DIR + "/**/*.wav", recursive=True):
        low = p.lower().replace("ı", "i").replace("ü", "u").replace("ö", "o").replace("ğ", "g")
        if "calm" in low or "sakin" in low: continue          # calm ~ neutral -> dropped (like our abstain)
        lab = next((v for k, v in TMAP.items() if k in low), None)
        if lab is None: continue
        stem = Path(p).stem
        rows.append({"id": "turev_" + stem, "path": p, "label": lab, "final_label": ID2LABEL[lab],
                     "weight": 1.0, "session": re.split(r"[_\-. ]", stem)[0] or stem})
    tv = pd.DataFrame(rows).drop_duplicates(subset="id", keep="first")
    if len(tv) < 200:
        raise RuntimeError(f"TurEV parse found only {len(tv)} usable wavs — inspect {TUREV_DIR} layout.")
    print("sample files:", [os.path.relpath(p, TUREV_DIR) for p in tv["path"].head(3)])
    print(f"TurEV: {len(tv)} unique clips | PZ={(tv['label'] == 1).mean():.0%}")
    # STRATIFIED 70/15/15 split: train / val (checkpoint selection) / held-out TurEV TEST.
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(tv))
    tr_ix, tmp_ix = train_test_split(idx, test_size=0.30, random_state=CONFIG["SEED"],
                                     stratify=tv["label"].values)
    va_ix, te_ix = train_test_split(tmp_ix, test_size=0.50, random_state=CONFIG["SEED"],
                                    stratify=tv["label"].values[tmp_ix])
    train_df, val_df = tv.iloc[tr_ix].reset_index(drop=True), tv.iloc[va_ix].reset_index(drop=True)
    turev_test_df = tv.iloc[te_ix].reset_index(drop=True)
    print(f"TurEV split: train={len(train_df)}  val={len(val_df)}  turev-test={len(turev_test_df)}")
else:
    raise ValueError("TRAIN_SOURCE must be 'ours' or 'turev'")

if CONFIG["TRAIN_SOURCE"] != "turev":
    turev_test_df = None

assert val_df["label"].nunique() == 2, \
    "validation split is single-class — split logic broken, do not train"

for name, d in (("train", train_df), ("val", val_df), ("test[OURS]", test_df)):
    print(f"{name:10s}: n={len(d):5d}  PZ={(d['final_label']=='PZ').mean():.0%}")

n_pz, n_ng = int((train_df.label == 1).sum()), int((train_df.label == 0).sum())
CLASS_W = torch.tensor([1.0, n_ng / max(n_pz, 1)], dtype=torch.float)
print(f"class weights: NG=1.0, PZ={CLASS_W[1]:.2f}")

In [ ]:
from transformers import AutoFeatureExtractor
try:
    FE = AutoFeatureExtractor.from_pretrained(CONFIG["BASE_MODEL"])
except Exception as e:
    print("no preprocessor config on the hub, using wav2vec2 defaults:", repr(e)[:120])
    from transformers import Wav2Vec2FeatureExtractor
    FE = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
                                  do_normalize=True, return_attention_mask=True)

class ClipDataset(torch.utils.data.Dataset):
    def __init__(self, frame, train):
        self.rows, self.train = frame.to_dict("records"), train
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        y, _ = librosa.load(r["path"], sr=16000, mono=True)
        y = y.astype(np.float32)
        max_len = int((CONFIG["TRAIN_CROP_S"] if self.train else CONFIG["EVAL_MAX_S"]) * 16000)
        if self.train and len(y) > max_len:
            st = np.random.randint(0, len(y) - max_len + 1); y = y[st:st + max_len]
        elif len(y) > max_len:
            y = y[-max_len:]                      
        if self.train and CONFIG["AUGMENT"] and np.random.rand() < 0.5:
            try:
                if np.random.rand() < 0.5:
                    y = librosa.effects.pitch_shift(y=y, sr=16000, n_steps=float(np.random.uniform(-2, 2)))
                else:
                    y = librosa.effects.time_stretch(y=y, rate=float(np.random.uniform(0.9, 1.11)))
                y = y.astype(np.float32)
            except Exception: pass
        return {"audio": y, "label": int(r["label"]), "sample_weight": float(r["weight"])}

def collate(batch):
    feats = FE([b["audio"] for b in batch], sampling_rate=16000, return_tensors="pt", padding=True)
    feats["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    feats["sample_weight"] = torch.tensor([b["sample_weight"] for b in batch], dtype=torch.float)
    return feats

ds_train, ds_val, ds_test = ClipDataset(train_df, True), ClipDataset(val_df, False), ClipDataset(test_df, False)
ds_turev_test = ClipDataset(turev_test_df, False) if turev_test_df is not None else None
print("datasets ready:", len(ds_train), len(ds_val), len(ds_test),
      f"(+ TurEV in-domain test: {len(ds_turev_test)})" if ds_turev_test else "")


In [ ]:
import torch.nn as nn
from transformers import (AutoModelForAudioClassification, TrainingArguments, Trainer,
                          EarlyStoppingCallback)
from sklearn.metrics import recall_score, f1_score, accuracy_score

def fresh_model():
    m = AutoModelForAudioClassification.from_pretrained(
        CONFIG["BASE_MODEL"], num_labels=2, label2id=LABEL2ID, id2label=ID2LABEL)
    try: m.freeze_feature_encoder()
    except Exception: pass
    return m

def compute_metrics(pred):
    y_true, y_hat = pred.label_ids, pred.predictions.argmax(-1)
    return {"uar": recall_score(y_true, y_hat, average="macro"),
            "f1_macro": f1_score(y_true, y_hat, average="macro"),
            "accuracy": accuracy_score(y_true, y_hat)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        w = inputs.pop("sample_weight")
        labels = inputs.pop("labels")
        out = model(**inputs)
        lf = nn.CrossEntropyLoss(weight=CLASS_W.to(out.logits.device), reduction="none")
        loss = (lf(out.logits, labels) * w).mean()
        return (loss, out) if return_outputs else loss

def drive_ok():
    return (not CONFIG["OUTPUT_DIR"].startswith("/content/drive")) or os.path.ismount("/content/drive")

from transformers import TrainerCallback
class DriveGuard(TrainerCallback):
    """Colab's Drive mount can die silently mid-run; then 'saves' land in a local ghost dir
    and vanish with the runtime. Abort loudly instead."""
    def on_save(self, args, state, control, **kwargs):
        if args.output_dir.startswith("/content/drive"):
            assert os.path.ismount("/content/drive"), \
                "DRIVE MOUNT LOST mid-run — outputs would be silently discarded."

def make_args(run_dir, seed):
    return TrainingArguments(
        output_dir=run_dir, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_uar", greater_is_better=True,
        num_train_epochs=CONFIG["EPOCHS"], learning_rate=CONFIG["LR"], warmup_ratio=0.1,
        per_device_train_batch_size=CONFIG["BATCH_TRAIN"], per_device_eval_batch_size=CONFIG["BATCH_EVAL"],
        bf16=torch.cuda.is_available(), logging_steps=20, save_total_limit=2,
        remove_unused_columns=False, dataloader_num_workers=2, report_to=[], seed=seed)

RUN_DIR = os.path.join(CONFIG["OUTPUT_DIR"], f"run_{TAG}")
model = fresh_model()
trainer = WeightedTrainer(model=model, args=make_args(RUN_DIR, CONFIG["SEED"]),
                          train_dataset=ds_train, eval_dataset=ds_val,
                          data_collator=collate, compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["EARLY_STOP_PATIENCE"]),
                                     DriveGuard()])
print("trainer ready — run dir:", RUN_DIR)


In [ ]:
if not CONFIG["RUN_CV"]:
    from transformers.trainer_utils import get_last_checkpoint
    ckpt = get_last_checkpoint(RUN_DIR) if os.path.isdir(RUN_DIR) else None
    if ckpt: print("resuming from", ckpt)
    try:
        trainer.train(resume_from_checkpoint=ckpt)
    except ValueError as e:
        if ckpt and "checkpoint" in str(e).lower():
            print("checkpoint unusable (partial/ghost dir on Drive) — starting fresh:", repr(e)[:120])
            import shutil; shutil.rmtree(RUN_DIR, ignore_errors=True)
            trainer.train()
        else:
            raise
    assert drive_ok(), "Drive mount lost — re-run Cell 1."
    best_dir = os.path.join(CONFIG["OUTPUT_DIR"], f"best_model_{TAG}")
    trainer.save_model(best_dir); FE.save_pretrained(best_dir)
    print("best model saved to", best_dir)
else:
    print("RUN_CV=True — skipping the single run; see the CV cell below.")


In [ ]:
if not CONFIG["RUN_CV"]:
    from sklearn.metrics import confusion_matrix

    def evaluate(ds, title):
        p = trainer.predict(ds)
        y_t = p.label_ids
        pr = torch.softmax(torch.tensor(p.predictions), dim=-1).numpy()
        y_h = pr.argmax(-1)
        def bci(fn, n_boot=2000, seed=42):
            rng = np.random.default_rng(seed); n = len(y_t); vals = []
            for _ in range(n_boot):
                ix = rng.integers(0, n, n)
                if len(set(y_t[ix])) < 2: continue
                vals.append(fn(y_t[ix], y_h[ix]))
            return np.percentile(vals, [2.5, 97.5])
        uar = recall_score(y_t, y_h, average="macro"); uci = bci(lambda a, b: recall_score(a, b, average="macro"))
        f1m = f1_score(y_t, y_h, average="macro");     fci = bci(lambda a, b: f1_score(a, b, average="macro"))
        acc = accuracy_score(y_t, y_h)
        cm = confusion_matrix(y_t, y_h)
        print(f"\n{title} (n={len(y_t)})")
        print(f"  UAR      : {uar:.3f}  (95% CI {uci[0]:.3f}-{uci[1]:.3f})   [chance = 0.500]")
        print(f"  macro-F1 : {f1m:.3f}  (95% CI {fci[0]:.3f}-{fci[1]:.3f})")
        print(f"  accuracy : {acc:.3f}   [majority-class baseline = {max(np.bincount(y_t))/len(y_t):.3f}]")
        print("  confusion (rows=true NG,PZ / cols=pred NG,PZ):"); print(" ", str(cm).replace("\n", "\n  "))
        return {"uar": uar, "uar_ci": list(uci), "f1_macro": f1m, "f1_ci": list(fci),
                "accuracy": acc, "confusion": cm.tolist(), "n": int(len(y_t))}, pr, y_h

    print(f"RUN TAG: {TAG}")
    main_stats, probs, y_hat = evaluate(ds_test, "TEST (unanimous-consensus labels, OUR corpus)")

    out = test_df[["id", "final_label", "speaker_role", "session"]].copy()
    out["pred"] = [ID2LABEL[i] for i in y_hat]; out["p_pz"] = probs[:, 1].round(4)
    out.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], f"predictions_{TAG}.csv"), index=False)

    results = {"tag": TAG, "train_source": CONFIG["TRAIN_SOURCE"], "base_model": CONFIG["BASE_MODEL"],
               "train_size": len(train_df), "test_ours": main_stats}
    if ds_turev_test is not None:
        turev_stats, _, _ = evaluate(ds_turev_test, "TUREV IN-DOMAIN TEST (held-out 15%)")
        results["test_turev"] = turev_stats

    assert drive_ok(), "Drive mount lost — re-run Cell 1 to remount, then re-run THIS cell only."
    json.dump(results, open(os.path.join(CONFIG["OUTPUT_DIR"], f"metrics_{TAG}.json"), "w"), indent=2)
    for fn in (f"predictions_{TAG}.csv", f"metrics_{TAG}.json"):
        fp = os.path.join(CONFIG["OUTPUT_DIR"], fn)
        print(f"verified on Drive: {fn} ({os.path.getsize(fp)} bytes)" if os.path.exists(fp)
              else f"!! MISSING: {fn}")

In [ ]:
if CONFIG["RUN_CV"]:
    from sklearn.model_selection import GroupKFold
    cv_df = df.copy()   # our full labeled set (1,837), regardless of TRAIN_SOURCE
    gkf = GroupKFold(n_splits=5)
    fold_res = []
    for k, (tr_ix, te_ix) in enumerate(gkf.split(cv_df, groups=cv_df["session"])):
        ftr_all, fte = cv_df.iloc[tr_ix], cv_df.iloc[te_ix]
        fte = fte[(fte["n_pz"] + fte["n_ng"]) >= 3]        # evaluate against >=3-vote labels only
        g2 = GroupShuffleSplit(n_splits=1, test_size=CONFIG["VAL_FRACTION"], random_state=CONFIG["SEED"])
        i1, i2 = next(g2.split(ftr_all, groups=ftr_all["session"]))
        ftr, fva = ftr_all.iloc[i1].reset_index(drop=True), ftr_all.iloc[i2].reset_index(drop=True)
        n_pz_f, n_ng_f = max(int((ftr.label == 1).sum()), 1), int((ftr.label == 0).sum())
        CLASS_W = torch.tensor([1.0, n_ng_f / n_pz_f], dtype=torch.float)   # rebinds the global used in the loss
        globals()["CLASS_W"] = CLASS_W
        t = WeightedTrainer(model=fresh_model(), args=make_args(f"/content/cv_fold{k}", CONFIG["SEED"] + k),
                            train_dataset=ClipDataset(ftr, True), eval_dataset=ClipDataset(fva, False),
                            data_collator=collate, compute_metrics=compute_metrics,
                            callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["EARLY_STOP_PATIENCE"])])
        t.train()
        p = t.predict(ClipDataset(fte.reset_index(drop=True), False))
        fold_uar = float(recall_score(p.label_ids, p.predictions.argmax(-1), average="macro"))
        fold_f1  = float(f1_score(p.label_ids, p.predictions.argmax(-1), average="macro"))
        fold_res.append({"fold": k, "uar": fold_uar, "f1_macro": fold_f1,
                         "n_test": int(len(fte)), "n_sessions_test": int(fte["session"].nunique())})
        print(f"fold {k}: UAR={fold_uar:.3f}  F1={fold_f1:.3f}  (n={len(fte)})")
        del t
        import gc; gc.collect(); torch.cuda.empty_cache()
    u = [f["uar"] for f in fold_res]
    print(f"\n================ CV SUMMARY ================")
    print(f"session-disjoint 5-fold UAR: {np.mean(u):.3f} +/- {np.std(u):.3f}  (folds: {[round(x,3) for x in u]})")
    json.dump({"base_model": CONFIG["BASE_MODEL"], "folds": fold_res,
               "mean_uar": float(np.mean(u)), "std_uar": float(np.std(u))},
              open(os.path.join(CONFIG["OUTPUT_DIR"], "cv_results.json"), "w"), indent=2)
    print("saved cv_results.json")